# Positional Encoding

Self-attention không có khái niệm thứ tự token (nó xử lý các token như một tập hợp không có thứ tự). Vì vậy cần cộng thêm một vector mã hoá vị trí (positional encoding) vào embedding đầu vào, để mô hình biết được vị trí tuyệt đối/tương đối của từng token trong chuỗi.

$$
PE(position, 2i) = \sin\left(\frac{position}{10000^{\frac{2i}{d_{model}}}}\right)
$$

$$
PE(position, 2i+1) = \cos\left(\frac{position}{10000^{\frac{2i}{d_{model}}}}\right)
$$

We can rewrite these as

$$
PE(position, i) = \sin\left(\frac{position}{10000^{\frac{i}{d_{model}}}}\right) \quad \text{when } i \text{ is even}
$$

$$
PE(position, i) = \cos\left(\frac{position}{10000^{\frac{i-1}{d_{model}}}}\right) \quad \text{when } i \text{ is odd}
$$

Cài đặt công thức trên bằng NumPy. Dùng d_model=6, max_sequence_length=10 (nhỏ) để dễ quan sát giá trị và shape.

In [1]:
import numpy as np

max_sequence_length = 10
d_model = 6

In [2]:
even_i = np.arange(0, d_model, 2).astype(np.float32)
even_i

array([0., 2., 4.], dtype=float32)

`even_i` là các chỉ số chiều chẵn (0, 2, 4, ...) — dùng để tính phần `sin` của PE.

In [3]:
even_dominator = np.power(10000, even_i / d_model)
even_dominator

array([  1.     ,  21.54435, 464.15897], dtype=float32)

Mẫu số `10000^(2i/d_model)` tăng theo cấp số nhân khi i tăng — chiều càng lớn thì tần số dao động của sin/cos càng thấp (bước sóng càng dài). Nhờ vậy các chiều khác nhau mã hoá vị trí ở nhiều "tần số" khác nhau, giúp mô hình phân biệt được vị trí gần lẫn xa.

In [4]:
odd_i = np.arange(1, d_model, 2).astype(np.float32)
odd_dominator = np.power(10000, (odd_i-1) / d_model)
print(odd_i)
print(odd_dominator)

[1. 3. 5.]
[  1.       21.54435 464.15897]


Dùng `odd_i - 1` để mẫu số của chiều lẻ (2i+1) trùng với mẫu số của chiều chẵn tương ứng (2i) — đúng như công thức gốc, cặp (sin, cos) cùng một chiều phải dùng chung tần số.

In [5]:
denominator = even_dominator 

`even_dominator` và `odd_dominator` giống hệt nhau (do cách tính ở trên), nên chỉ cần giữ lại một biến `denominator` dùng chung.

In [6]:
position = np.arange(max_sequence_length, dtype=np.float32).reshape(max_sequence_length, 1)

In [7]:
position

array([[0.],
       [1.],
       [2.],
       [3.],
       [4.],
       [5.],
       [6.],
       [7.],
       [8.],
       [9.]], dtype=float32)

In [8]:
even_PE = np.sin(position / denominator)
odd_PE = np.cos(position / denominator)

`even_PE` dùng `sin` cho các chiều chẵn, `odd_PE` dùng `cos` cho các chiều lẻ — cùng `denominator` nên mỗi cặp chiều (2i, 2i+1) có cùng tần số, chỉ khác pha (sin vs cos).

In [9]:
even_PE

array([[ 0.        ,  0.        ,  0.        ],
       [ 0.841471  ,  0.04639922,  0.00215443],
       [ 0.9092974 ,  0.09269849,  0.00430886],
       [ 0.14112   ,  0.13879807,  0.00646326],
       [-0.7568025 ,  0.18459871,  0.00861763],
       [-0.9589243 ,  0.23000169,  0.01077196],
       [-0.2794155 ,  0.27490923,  0.01292625],
       [ 0.6569866 ,  0.31922463,  0.01508047],
       [ 0.98935825,  0.36285236,  0.01723462],
       [ 0.41211846,  0.4056985 ,  0.01938869]], dtype=float32)

In [10]:
odd_PE

array([[ 1.        ,  1.        ,  1.        ],
       [ 0.5403023 ,  0.998923  ,  0.9999977 ],
       [-0.4161468 ,  0.9956942 ,  0.9999907 ],
       [-0.9899925 ,  0.9903207 ,  0.99997914],
       [-0.6536436 ,  0.98281395,  0.99996287],
       [ 0.28366217,  0.97319025,  0.999942  ],
       [ 0.96017027,  0.9614702 ,  0.99991643],
       [ 0.75390226,  0.9476791 ,  0.9998863 ],
       [-0.14550003,  0.9318466 ,  0.99985147],
       [-0.91113025,  0.91400695,  0.999812  ]], dtype=float32)

In [11]:
stacked = np.stack((even_PE, odd_PE), axis=2)
stacked.shape

(10, 3, 2)

`np.stack` rồi `reshape` để xen kẽ sin/cos đúng thứ tự theo chỉ số chiều: `[sin0, cos0, sin1, cos1, sin2, cos2, ...]`, khớp với công thức PE(pos, 2i) = sin, PE(pos, 2i+1) = cos.

In [12]:
PE = stacked.reshape(stacked.shape[0], -1)
PE

array([[ 0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         1.        ],
       [ 0.841471  ,  0.5403023 ,  0.04639922,  0.998923  ,  0.00215443,
         0.9999977 ],
       [ 0.9092974 , -0.4161468 ,  0.09269849,  0.9956942 ,  0.00430886,
         0.9999907 ],
       [ 0.14112   , -0.9899925 ,  0.13879807,  0.9903207 ,  0.00646326,
         0.99997914],
       [-0.7568025 , -0.6536436 ,  0.18459871,  0.98281395,  0.00861763,
         0.99996287],
       [-0.9589243 ,  0.28366217,  0.23000169,  0.97319025,  0.01077196,
         0.999942  ],
       [-0.2794155 ,  0.96017027,  0.27490923,  0.9614702 ,  0.01292625,
         0.99991643],
       [ 0.6569866 ,  0.75390226,  0.31922463,  0.9476791 ,  0.01508047,
         0.9998863 ],
       [ 0.98935825, -0.14550003,  0.36285236,  0.9318466 ,  0.01723462,
         0.99985147],
       [ 0.41211846, -0.91113025,  0.4056985 ,  0.91400695,  0.01938869,
         0.999812  ]], dtype=float32)

In [13]:
class PositionalEncoding:
    def __init__(self, d_model, max_sequence_length):
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length

    def forward(self):
        even_i = np.arange(0, self.d_model, 2).astype(np.float32)
        even_dominator = np.power(10000, even_i / self.d_model)

        odd_i = np.arange(1, self.d_model, 2).astype(np.float32)
        odd_dominator = np.power(10000, (odd_i - 1) / self.d_model)

        denominator = even_dominator

        position = np.arange(self.max_sequence_length, dtype=np.float32).reshape(self.max_sequence_length, 1)

        even_PE = np.sin(position / denominator)
        odd_PE = np.cos(position / denominator)

        stacked = np.stack((even_PE, odd_PE), axis=2)
        PE = stacked.reshape(stacked.shape[0], -1)

        return PE

Đóng gói thành class `PositionalEncoding` để tái sử dụng. Output là ma trận PE shape `(max_sequence_length, d_model)`, cộng trực tiếp vào embedding đầu vào (cùng shape theo từng token) trước khi đưa vào encoder/decoder.

In [14]:
pe = PositionalEncoding(d_model = 6 , max_sequence_length = 10)
pe.forward()

array([[ 0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         1.        ],
       [ 0.841471  ,  0.5403023 ,  0.04639922,  0.998923  ,  0.00215443,
         0.9999977 ],
       [ 0.9092974 , -0.4161468 ,  0.09269849,  0.9956942 ,  0.00430886,
         0.9999907 ],
       [ 0.14112   , -0.9899925 ,  0.13879807,  0.9903207 ,  0.00646326,
         0.99997914],
       [-0.7568025 , -0.6536436 ,  0.18459871,  0.98281395,  0.00861763,
         0.99996287],
       [-0.9589243 ,  0.28366217,  0.23000169,  0.97319025,  0.01077196,
         0.999942  ],
       [-0.2794155 ,  0.96017027,  0.27490923,  0.9614702 ,  0.01292625,
         0.99991643],
       [ 0.6569866 ,  0.75390226,  0.31922463,  0.9476791 ,  0.01508047,
         0.9998863 ],
       [ 0.98935825, -0.14550003,  0.36285236,  0.9318466 ,  0.01723462,
         0.99985147],
       [ 0.41211846, -0.91113025,  0.4056985 ,  0.91400695,  0.01938869,
         0.999812  ]], dtype=float32)